In [23]:
import pandas as pd
import numpy as np
import os
import json as jsonlib
from rapidfuzz import process, fuzz
import unicodedata
import re
import shutil
import warnings

warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)

# Obtenemos el CSV con competiciones
cdir = os.getcwd()
utils = os.path.join(os.path.abspath(os.path.join(cdir, '..', '..')), 'utils')
comps = pd.read_csv(os.path.join(utils, 'comps.csv'), sep=';')

# JSON con temporadas deseadas
with open(os.path.join(utils, 'des_seasons.json'), 'r', encoding='utf-8') as f:
    desired_seasons = jsonlib.load(f)

# Creación de slug
def create_slug(text: str) -> str:

    # Letra minuscula
    text = text.lower()
    
    # Elminar acentos
    text = ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')
    
    # Substituir espacios por '-'
    text = re.sub(r"\s+", "_", text)
    
    # Eliminar caracteres no-alfanumericos (menos -)
    text = re.sub(r"[^a-z0-9_]", "", text)
    text = re.sub(r"_+", "_", text).strip("_")
    
    return text

# Unifica los nombres de los equipos usando similitud de los strings
def unify_teams(teams_fm, teams_sw, teams_ss, threshold=40):
    
    # Lista
    unified = []

    # Para cada equipo en fm
    for team in sorted(teams_fm):
        
        # Buscar mejor match en sw y en ss
        match_sw = process.extractOne(team, teams_sw, scorer=fuzz.token_sort_ratio)
        match_ss = process.extractOne(team, teams_ss, scorer=fuzz.token_sort_ratio)
        
        # Match si sigue el threshold
        sw_name = match_sw[0] if match_sw and match_sw[1] >= threshold else None
        ss_name = match_ss[0] if match_ss and match_ss[1] >= threshold else None
        
        unified.append({"fm": team, "sw": sw_name, "ss": ss_name})
    
    return pd.DataFrame(unified)

# Función para unificar tablas de classificación
def unify_standing_tables(fm: pd.DataFrame, sw: pd.DataFrame, ss: pd.DataFrame, ht_sw: pd.DataFrame, fm_to_sw: dict, ss_to_sw: dict, sw_to_sw: dict) -> pd.DataFrame:

    # Nombre de equipo con la normalización
    fm['team'] = fm['team'].map(fm_to_sw)
    ss['team'] = ss['team'].map(ss_to_sw)
    sw['team'] = sw['team'].map(sw_to_sw)
    ht_sw['team'] = ht_sw['team'].map(sw_to_sw)

    # Unión de los tres dataframes a partir de la columna 'team'
    standings_df = (sw.merge(fm, on='team', how='inner', suffixes=('_sw', '_fm')).merge(ss, on='team', how='inner', suffixes=('', '_ss')))
    standings_df = standings_df.merge(ht_sw, on='team', how='inner', suffixes=('', '_ht'))

    # Columnas a sacar
    cols_to_drop = ["league_fm", "season_fm", "pos", "pts", "played", "wins_fm", "draws_fm", "losses_fm", "goalsScored", "goalsConceded", "goalDiff",
                    "league", "season", "position", "team_slug", "promotion", "points_ss", "matches", "wins", "losses", "draws", "scores_for", "scores_against",
                    "league_ht", "season_ht", "matches_played_ht"]
    standings_df = standings_df.drop(columns=cols_to_drop)

    # Renombrar columnas
    rename_dict = {"league_sw":"League","season_sw":"Season","rank":"Position","status":"Status","team":"Team","points":"Points","matches_played":"Matches","wins_sw":"Wins",
                   "draws_sw":"Draws","losses_sw":"Losses","goals_for":"GoalsFor","goals_against":"GoalsAgainst","home_pos":"HomePosition","home_pts":"HomePoints","home_played":"HomeMatches",
                   "home_wins":"HomeWins","home_draws":"HomeDraws","home_losses":"HomeLosses","home_goalsScored":"HomeGoalsFor","home_goalsConceded":"HomeGoalsAgainst","home_goalDiff":"HomeGoalsDiff",
                   "away_pos":"AwayPosition","away_pts":"AwayPoints","away_played":"AwayMatches","away_wins":"AwayWins","away_draws":"AwayDraws","away_losses":"AwayLosses","away_goalsScored":"AwayGoalsFor",
                   "away_goalsConceded":"AwayGoalsAgainst","away_goalDiff":"AwayGoalsDiff","xG":"xG","xGA":"xGA","xGDiff":"xGDiff","xGADiff":"xGADiff","xPos":"xPosition","xPosDiff":"xPositionDiff",
                   "xPts":"xPoints","xPtsDiff":"xPointsDiff","rank_ht":"HalfTimePosition","points_ht":"HalfTimePoints","wins_ht":"HalfTimeWins","draws_ht":"HalfTimeDraws","losses_ht":"HalfTimeLosses",
                   "goals_for_ht":"HalfTimeGoalsFor","goals_against_ht":"HalfTimeGoalsAgainst"}
    standings_df = standings_df.rename(columns=rename_dict)

    # Ordenar df
    ordered_cols = ["League", "Season", "Position", "Team", "Status", "Matches", "Points", "Wins", "Draws", "Losses", "GoalsFor", "GoalsAgainst", "HomePosition", "HomeMatches", 
                    "HomePoints", "HomeWins", "HomeDraws", "HomeLosses", "HomeGoalsFor", "HomeGoalsAgainst", "HomeGoalsDiff", "AwayPosition", "AwayMatches", "AwayPoints",
                    "AwayWins", "AwayDraws", "AwayLosses", "AwayGoalsFor", "AwayGoalsAgainst", "AwayGoalsDiff", "HalfTimePosition", "HalfTimePoints", "HalfTimeWins", "HalfTimeDraws",
                    "HalfTimeLosses", "HalfTimeGoalsFor", "HalfTimeGoalsAgainst", "xG", "xGA", "xGDiff", "xGADiff", "xPosition", "xPositionDiff", "xPoints", "xPointsDiff"]
    standings_df = standings_df[ordered_cols]

    # Columnas a comprovar y añadir
    standings_df.insert(standings_df.columns.get_loc("GoalsAgainst") + 1, "GoalsDiff", standings_df["GoalsFor"] - standings_df["GoalsAgainst"])
    standings_df['HomeGoalsDiff'] = standings_df['HomeGoalsFor'] - standings_df['HomeGoalsAgainst']
    standings_df['AwayGoalsDiff'] = standings_df['AwayGoalsFor'] - standings_df['AwayGoalsAgainst']
    standings_df['xGDiff'] = standings_df['GoalsFor'] - standings_df['xG']
    standings_df['xGADiff'] = standings_df['GoalsAgainst'] - standings_df['xGA']
    standings_df['xPositionDiff'] = standings_df['Position'] - standings_df['xPosition']
    standings_df['xPointsDiff'] = standings_df['Points'] - standings_df['xPoints']

    return standings_df

# Trato del df de standings de attendance
def attendance_standings_cleaner(attendance_st: pd.DataFrame, teams_dict: dict) -> pd.DataFrame:

    # Equipo al nombre unificado
    attendance_st['team'] = attendance_st['team'].map(teams_dict)

    # Cambio de nombres
    rename_cols = {'league':'League', 'season':'Season', 'rank': 'AttendancePosition', 'team':'Team', 'venue_name':'Venue', 'min_attendance':'MinAttendance',
                   'max_attendance':'MaxAttendance', 'total_attendance':'TotalAttendance', 'avg_attendance':'AvgAttendance', 'capacity':'Capacity', 'percent_sold':'PercentSold'}
    attendance_st = attendance_st.rename(columns=rename_cols)

    # Orden
    order_cols = ['League', 'Season', 'AttendancePosition', 'Team', 'Venue', 'MinAttendance', 'MaxAttendance', 'TotalAttendance', 'AvgAttendance', 'Capacity', 'PercentSold']
    attendance_st = attendance_st[order_cols]

    return attendance_st

# Función para obtener la información sobre el estadio
def venue_information(attendance_df: pd.DataFrame, venues_df: pd.DataFrame, raw_data_images_path: str, clean_data_images_path: str) -> pd.DataFrame:

    # Creación de la carpeta de imagenes si no existe
    os.makedirs(clean_data_images_path, exist_ok=True)

    # Obtenemos los nombres de los campos juntos
    venues_a = attendance_df['Venue'].tolist()
    venues_b = venues_df['name'].tolist()

    # Para cada campo
    dict_venues = {}
    for venue in (venues_b):
        venue_matched = process.extractOne(venue, venues_a, scorer=fuzz.token_sort_ratio)
        venue_name = venue_matched[0] if venue_matched and venue_matched[1] >= 0.2 else None
        dict_venues[venue] = venue_name

    # Aplicamos a venues info
    venues_df['Venue'] = venues_df['name'].map(dict_venues)

    # Merge
    final_venues_df = venues_df.merge(attendance_df, on='Venue', how='inner')

    # Columnas a sacar
    cols_to_drop = ['league', 'season', 'name', 'slug', 'capacity', 'matches']
    final_venues_df = final_venues_df.drop(columns=cols_to_drop)

    # Columnas a renombrar
    rename_dict = {"id":"Id", "city":"City", "country":"Country", "latitude":"Latitude", "longitude":"Longitude", "home_goals":"HomeGoals", "away_goals":"AwayGoals", "avg_red_cards_game":"AvgRedCardsGame",
                   "avg_ck_game":"AvgCornersGame", "home_wins_perc":"HomeWinsPerc", "away_wins_perc":"AwayWinsPerc", "draws_perc":"DrawsPerc"}
    final_venues_df = final_venues_df.rename(columns=rename_dict)

    # Slug
    final_venues_df["Slug"] = final_venues_df["Venue"].apply(create_slug)

    # Orden
    ordered_cols = ["League", "Season", "Id", "Venue", "Slug", "City", "Country", "Latitude", "Longitude", "Team", "Capacity", "MinAttendance", "MaxAttendance", "AvgAttendance", "TotalAttendance", 
                    "PercentSold", "AttendancePosition", "HomeGoals", "AwayGoals", "HomeWinsPerc", "AwayWinsPerc", "DrawsPerc", "AvgRedCardsGame", "AvgCornersGame"]
    final_venues_df = final_venues_df[ordered_cols]

    # Vamos a la carpeta con las imagenes de campos para moverlos
    for _, row in final_venues_df.iterrows():

        # Paths a partir de los nombres
        old_path = os.path.join(raw_data_images_path, f'{row['Id']}.png')
        new_path = os.path.join(clean_data_images_path, f'{row['Slug']}.png')

        # Mover fichero al nuevo destino
        if os.path.exists(old_path):
            shutil.copy(old_path, new_path)

    # Sacamos ID
    final_venues_df = final_venues_df.drop(columns=['Id'])

    return final_venues_df.sort_values(by='Venue', ignore_index=True)

# Unificar información de equipos
def teams_information(ss: pd.DataFrame, sw: pd.DataFrame, ss_to_sw: dict, raw_data_images_path: str, clean_data_images_path: str) -> pd.DataFrame:

    # Creación de la carpeta de imagenes si no existe
    os.makedirs(clean_data_images_path, exist_ok=True)

    # Cambiar el nombre del equipo en SS
    ss['club_name'] = ss['name'].map(ss_to_sw)

    # Merge
    teams_df = ss.merge(sw, on='club_name', how='inner', suffixes=['_ss', '_sw'])

    # Columnas a sacar
    cols_to_drop = ['name_ss', 'slug_ss', 'short_name_ss', 'full_name', 'venue_ss', 'league_sw', 'season_sw', 'id_sw', 'code_sw', 'slug_sw']
    teams_df = teams_df.drop(columns=cols_to_drop)

    # Renombrar
    rename_dict = {"league_ss":"League", "season_ss":"Season", "id_ss":"TeamId", "code_ss":"Abbreviation", "manager":"Manager", "country":"Country", "primary_colour":"PrimaryColour", "secondary_colour":"SecondaryColour",
                   "text_colour":"TextColour", "foundation":"FoundationYear", "club_name":"Team", "name_sw":"LongName", "short_name_sw":"ShortName", "venue_sw":"Venue"}
    teams_df = teams_df.rename(columns=rename_dict)

    # Slug
    teams_df["Slug"] = teams_df["Team"].apply(create_slug)
    
    # Reordenar
    ordered_cols = ["League", "Season", "TeamId", "Abbreviation", "Team", "Slug", "LongName", "ShortName", "Country", "FoundationYear", "Manager", "PrimaryColour", "SecondaryColour", "TextColour", "Venue"]
    teams_df = teams_df[ordered_cols]

    # Editamos columnas
    teams_df["FoundationYear"] = (pd.to_datetime(teams_df["FoundationYear"], unit="s", errors="coerce").dt.strftime("%d/%m/%Y"))
    teams_df["PrimaryColour"] = teams_df["PrimaryColour"].str.upper()
    teams_df["SecondaryColour"] = teams_df["SecondaryColour"].str.upper()
    teams_df["TextColour"] = teams_df["TextColour"].str.upper()

    # Vamos a la carpeta con las imagenes de escudos para moverlos
    for _, row in teams_df.iterrows():

        # Paths a partir de los nombres
        old_path = os.path.join(raw_data_images_path, f'{row['TeamId']}.png')
        new_path = os.path.join(clean_data_images_path, f'{row['Slug']}.png')

        # Mover fichero al nuevo destino
        if os.path.exists(old_path):
            shutil.copy(old_path, new_path)

    # Sacamos ID
    teams_df = teams_df.drop(columns=['TeamId'])

    # Ordenado por equipo
    return teams_df.sort_values(by='Team', ignore_index=True)

# Información de managers
def managers_information(teams_df: pd.DataFrame, ss: pd.DataFrame, sw: pd.DataFrame, ss_to_sw: dict, sw_to_sw: dict, raw_data_images_path: str, clean_data_images_path: str) -> pd.DataFrame:

    # Creación de la carpeta de imagenes si no existe
    os.makedirs(clean_data_images_path, exist_ok=True)

    # Mapear equipo
    ss['team'] = ss['team'].map(ss_to_sw)
    sw['team'] = sw['team'].map(sw_to_sw)

    # Lista para concatenar manager info
    managers_list = []
    staff_list = []

    # Por cada equipo, buscamos info en managers
    for _, row in teams_df.iterrows():

        # Info que necesitaremos
        manager = row['Manager']
        team = row['Team']

        # Info manager de Sofascore - sin tener que hacer comparación
        manager_info_ss = ss[ss['name'] == manager]
        if not manager_info_ss.empty:
            manager_dict_ss = manager_info_ss.iloc[0].to_dict()
            manager_dict_ss["manager_id"] = manager_dict_ss.pop("id")
        else:
            manager_dict_ss = {}

        # Info manager de Scoresway - comparación
        subset_managers = sw[(sw['team'] == team) & (sw['type'] == 'coach')]['short_name'].tolist()
        match_manager = process.extractOne(manager, subset_managers, scorer=fuzz.token_sort_ratio)
        manager_name_sw = match_manager[0] if match_manager and match_manager[1] >= 0.2 else None

        # Filtramos en el dataframe
        manager_info_sw = sw[sw['short_name'] == manager_name_sw]
        if not manager_info_sw.empty:
            manager_dict_sw = manager_info_sw.iloc[0].to_dict()
        else:
            manager_dict_sw = {}

        # Equipo
        manager_dict = manager_dict_ss | manager_dict_sw
        manager_dict['def_team'] = team
        managers_list.append(manager_dict)

        # Resta de staff
        other_staff = sw[(sw['team'] == team) & (sw['type'] != 'coach')]
        other_staff['def_team'] = team
        staff_list.append(other_staff)

    # Dataframe
    managers_df = pd.DataFrame(managers_list)
    staff_df = pd.concat(staff_list, ignore_index=True)
    all_staff_df = pd.concat([managers_df, staff_df], ignore_index=True)

    # Quitar columnas
    cols_to_drop = ['id', 'slug', 'team', 'country', 'first_name', 'last_name']
    all_staff_df = all_staff_df.drop(columns=cols_to_drop)

    # Rename
    rename_dict = {"league": "League", "season": "Season", "manager_id": "ManagerId", "name": "Manager", "short_name": "ShortName", "pref_formation": "PreferredFormation", "date_birth": "BirthDate", "matches": "Matches", "wins": "Wins", 
                   "draws": "Draws", "losses": "Losses", "goals_for": "GoalsFor", "goals_against": "GoalsAgainst", "points": "Points", "match_name": "MatchName", "nationality": "Nationality", "type": "Role", "def_team": "Team"}
    all_staff_df = all_staff_df.rename(columns=rename_dict)

    # Slug
    all_staff_df['Slug'] = all_staff_df['Manager'].apply(create_slug)

    # Ordenar
    ordered_cols = ["League", "Season", "ManagerId", "Manager", "Slug", "ShortName", "MatchName", "BirthDate", "Nationality", "Role", "PreferredFormation", "Team", "Matches", "Wins", "Draws", "Losses", "GoalsFor", "GoalsAgainst", "Points"]
    all_staff_df = all_staff_df[ordered_cols]

    # Tratado de columnas
    all_staff_df["BirthDate"] = (pd.to_datetime(all_staff_df["BirthDate"], unit="s", errors="coerce").dt.strftime("%d/%m/%Y"))
    all_staff_df["ManagerId"] = pd.to_numeric(all_staff_df["ManagerId"], errors="coerce").astype("Int64")
    all_staff_df["Matches"] = pd.to_numeric(all_staff_df["Matches"], errors="coerce").astype("Int64")
    all_staff_df["Wins"] = pd.to_numeric(all_staff_df["Wins"], errors="coerce").astype("Int64")
    all_staff_df["Draws"] = pd.to_numeric(all_staff_df["Draws"], errors="coerce").astype("Int64")
    all_staff_df["Losses"] = pd.to_numeric(all_staff_df["Losses"], errors="coerce").astype("Int64")
    all_staff_df["GoalsFor"] = pd.to_numeric(all_staff_df["GoalsFor"], errors="coerce").astype("Int64")
    all_staff_df["GoalsAgainst"] = pd.to_numeric(all_staff_df["GoalsAgainst"], errors="coerce").astype("Int64")
    all_staff_df["Points"] = pd.to_numeric(all_staff_df["Points"], errors="coerce").astype("Int64")
    all_staff_df["Role"] = np.where(all_staff_df["Role"] == "coach", "Coach", "Assistant")

    # Vamos a la carpeta con las imagenes de fotos de entrenadores para moverlos
    for _, row in all_staff_df.iterrows():

        # Paths a partir de los nombres
        old_path = os.path.join(raw_data_images_path, f'{row['ManagerId']}.png')
        new_path = os.path.join(clean_data_images_path, f'{row['Slug']}.png')

        # Mover fichero al nuevo destino
        if os.path.exists(old_path):
            shutil.copy(old_path, new_path)

    all_staff_df = all_staff_df.drop(columns=['ManagerId'])

    return all_staff_df.sort_values(by="Manager", ignore_index=True)

# Creación de dataframe de información de jugadores
def players_information(ss: pd.DataFrame, ss_to_sw: dict, raw_data_images_path: str, clean_data_images_path: str) -> pd.DataFrame:

    # Creación de la carpeta de imagenes si no existe
    os.makedirs(clean_data_images_path, exist_ok=True)

    players_info = ss.copy()

    # Mapeo de equipos
    players_info['team'] = players_info['team'].map(ss_to_sw)
    players_info = players_info.dropna(subset=['team'])

    # Rename de columnas
    rename_dict = {'league': 'League', 'season': 'Season', 'id': 'PlayerId', 'name': 'Player', 'slug': 'Slug', 'short_name': 'ShortName', 'team': 'Team',
                   'country': 'Country', 'position': 'PrimaryPosition', 'second_position': 'SecondaryPosition', 'third_position': 'ThirdPosition', 'weight': 'Weight',
                   'height': 'Height', 'shirt_number': 'ShirtNumber', 'pref_foot': 'PreferredFoot', 'date_birth': 'BirthDate', 'contract_until': 'ContractUntil', 'market_value': 'MarketValue'}
    players_info = players_info.rename(columns=rename_dict)

    # Tratado de columnas
    players_info['Slug'] = players_info['Player'].apply(create_slug)
    players_info['Position'] = (players_info[['PrimaryPosition', 'SecondaryPosition', 'ThirdPosition']].bfill(axis=1).iloc[:, 0])
    players_info['Height'] = (pd.to_numeric(players_info['Height'], errors='coerce').replace(0, pd.NA).astype('Int64'))
    players_info['Weight'] = (pd.to_numeric(players_info['Weight'], errors='coerce').replace(0, pd.NA).astype('Int64'))
    players_info["BirthDate"] = (pd.to_datetime(players_info["BirthDate"], unit="s", errors="coerce").dt.strftime("%d/%m/%Y"))
    players_info["ContractUntil"] = (pd.to_datetime(players_info["ContractUntil"], unit="s", errors="coerce").dt.strftime("%d/%m/%Y"))
    players_info['MarketValue'] = players_info['MarketValue'] / 1000000

    # Reordenado
    new_order = ['League', 'Season', 'PlayerId', 'Player', 'ShortName', 'Slug', 'Team', 'Country', 'Position',
                'ShirtNumber', 'PreferredFoot', 'Height', 'Weight', 'BirthDate', 'ContractUntil', 'MarketValue']
    players_info = players_info[new_order]

    # Vamos a la carpeta con las imagenes de escudos para moverlos
    for _, row in players_info.iterrows():

        # Paths a partir de los nombres
        old_path = os.path.join(raw_data_images_path, f'{row['PlayerId']}.png')
        new_path = os.path.join(clean_data_images_path, f'{row['Slug']}.png')

        # Mover fichero al nuevo destino
        if os.path.exists(old_path):
            shutil.copy(old_path, new_path)


    players_info = players_info.drop(columns=['PlayerId'])

    return players_info.sort_values(['Team', 'Player'], ignore_index=True)

In [24]:
# Unifica la información entera de una liga en una temporada
def unify_league_season_data(league_id: int, season: str, raw_data_path: str, proc_data_path: str, clean_data_path: str) -> None:

    # Obtenemos el nombre de la liga
    league_name = comps[comps['id'] == league_id]['tournament'].iloc[0]
    league_slug = league_name.lower().replace(' ', '-')

    # Creación de la carpeta para añadir los datos limpios
    clean_league_path = os.path.join(clean_data_path, league_slug, season)
    os.makedirs(clean_league_path, exist_ok=True)

    # Carpeta con información
    info_path = os.path.join(clean_league_path, 'info')
    os.makedirs(info_path, exist_ok=True)

    # Path de la liga según fuente
    proc_league_path_ss = os.path.join(proc_data_path, 'ss', league_slug)
    proc_league_path_sw = os.path.join(proc_data_path, 'sw', league_slug)
    proc_league_path_fm = os.path.join(proc_data_path, 'fm', league_slug)

    # Obtención de los CSV que contienen los nombres de equipos
    teams_fm_df = pd.read_csv(os.path.join(proc_league_path_fm, 'standings', f'{season}.csv'), sep=';')         # Son standings - no tenemos info equipos
    teams_sw_df = pd.read_csv(os.path.join(proc_league_path_sw, season, 'info', 'teams.csv'), sep=';')
    teams_ss_df = pd.read_csv(os.path.join(proc_league_path_ss, season, 'info', 'team.csv'), sep=';')

    # Nombres equipos
    teams_fm = teams_fm_df['team'].unique().tolist()
    teams_sw = teams_sw_df['club_name'].unique().tolist()
    teams_ss = teams_ss_df['name'].unique().tolist()

    # Obtención del dataframe con los equipos unificados por nombre
    teams_unified_df = unify_teams(teams_fm, teams_sw, teams_ss, threshold=40)  
    fm_team_to_sw = teams_unified_df.set_index('fm')['sw'].to_dict()
    ss_team_to_sw = teams_unified_df.set_index('ss')['sw'].to_dict()
    sw_team_to_sw = teams_sw_df.set_index('name')['club_name'].to_dict()

    # Tratado de tablas de clasificación
    standings_fm = pd.read_csv(os.path.join(proc_league_path_fm, 'standings', f'{season}.csv'), sep=';')
    standings_sw = pd.read_csv(os.path.join(proc_league_path_sw, season, 'info', 'standings_total.csv'), sep=';')
    standings_ss = pd.read_csv(os.path.join(proc_league_path_ss, season, 'info', 'standings', 'total.csv'), sep=';')

    # Tablas de clasificación especiales de SW
    ht_standings_sw = pd.read_csv(os.path.join(proc_league_path_sw, season, 'info', 'standings_halftime.csv'), sep=';')
    att_standings_sw = pd.read_csv(os.path.join(proc_league_path_sw, season, 'info', 'standings_attendance.csv'), sep=';')

    # Pasamos por la función
    standings_df = unify_standing_tables(fm=standings_fm, sw=standings_sw, ss=standings_ss, ht_sw=ht_standings_sw, fm_to_sw=fm_team_to_sw, ss_to_sw=ss_team_to_sw, sw_to_sw=sw_team_to_sw)
    standings_df.to_csv(os.path.join(info_path, 'Standings.csv'), sep=';', index=False)

    # Tratado de dataframe de standings de attendance con el de información de estadios
    attendance_df = attendance_standings_cleaner(attendance_st=att_standings_sw, teams_dict=sw_team_to_sw)
    venues_info = pd.read_csv(os.path.join(proc_league_path_ss, season, 'info', 'venue.csv'), sep=';')
    venues_df = venue_information(attendance_df=attendance_df, venues_df=venues_info, raw_data_images_path=os.path.join(raw_data_path, 'ss', league_slug, season, 'images', 'venue'), clean_data_images_path=os.path.join(clean_league_path, 'images', 'venues'))
    venues_df.to_csv(os.path.join(info_path, 'Venues.csv'), sep=';', index=False)

    # Tratado de información de equipos
    teams_df_ss = pd.read_csv(os.path.join(proc_league_path_ss, season, 'info', 'team.csv'), sep=';')
    teams_df_sw = pd.read_csv(os.path.join(proc_league_path_sw, season, 'info', 'teams.csv'), sep=';')
    teams_df = teams_information(ss=teams_df_ss, sw=teams_df_sw, ss_to_sw=ss_team_to_sw, raw_data_images_path=os.path.join(raw_data_path, 'ss', league_slug, season, 'images', 'team'), clean_data_images_path=os.path.join(clean_league_path, 'images', 'teams'))
    teams_df.to_csv(os.path.join(info_path, 'Teams.csv'), sep=';', index=False)

    # Tratado de información de managers
    managers_df_ss = pd.read_csv(os.path.join(proc_league_path_ss, season, 'info', 'manager.csv'), sep=';')
    managers_df_sw = pd.read_csv(os.path.join(proc_league_path_sw, season, 'info', 'managers.csv'), sep=';')
    managers_df = managers_information(teams_df=teams_df, ss=managers_df_ss, sw=managers_df_sw, ss_to_sw=ss_team_to_sw, sw_to_sw=sw_team_to_sw, raw_data_images_path=os.path.join(raw_data_path, 'ss', league_slug, season, 'images', 'manager'), clean_data_images_path=os.path.join(clean_league_path, 'images', 'managers'))
    managers_df.to_csv(os.path.join(info_path, 'Managers.csv'), sep=';', index=False)

    # Tratado de dataframe de jugadores - en este caso solo usamos el de Sofascore
    players_df_ss = pd.read_csv(os.path.join(proc_league_path_ss, season, 'info', 'player.csv'), sep=';')
    players_df = players_information(ss=players_df_ss, ss_to_sw=ss_team_to_sw, raw_data_images_path=os.path.join(raw_data_path, 'ss', league_slug, season, 'images', 'player'), clean_data_images_path=os.path.join(clean_league_path, 'images', 'players'))
    players_df.to_csv(os.path.join(info_path, 'Players.csv'), sep=';', index=False)

In [25]:
league_id = 73
season = '2526'
raw_data_path = r'G:\Xavier Rosinach Capell\03 Máster en Big Data Deportivo\04 Trabajo Final de Master\data\raw'
proc_data_path = r'G:\Xavier Rosinach Capell\03 Máster en Big Data Deportivo\04 Trabajo Final de Master\data\proc'
clean_data_path = r'G:\Xavier Rosinach Capell\03 Máster en Big Data Deportivo\04 Trabajo Final de Master\data\clean'

# Obtenemos el nombre de la liga
league_name = comps[comps['id'] == league_id]['tournament'].iloc[0]
league_slug = league_name.lower().replace(' ', '-')

# Creación de la carpeta para añadir los datos limpios
clean_league_path = os.path.join(clean_data_path, league_slug, season)
os.makedirs(clean_league_path, exist_ok=True)

# Carpeta con información
info_path = os.path.join(clean_league_path, 'info')
os.makedirs(info_path, exist_ok=True)

# Path de la liga según fuente
proc_league_path_ss = os.path.join(proc_data_path, 'ss', league_slug)
proc_league_path_sw = os.path.join(proc_data_path, 'sw', league_slug)
proc_league_path_fm = os.path.join(proc_data_path, 'fm', league_slug)

# Obtención de los CSV que contienen los nombres de equipos
teams_fm_df = pd.read_csv(os.path.join(proc_league_path_fm, 'standings', f'{season}.csv'), sep=';')         # Son standings - no tenemos info equipos
teams_sw_df = pd.read_csv(os.path.join(proc_league_path_sw, season, 'info', 'teams.csv'), sep=';')
teams_ss_df = pd.read_csv(os.path.join(proc_league_path_ss, season, 'info', 'team.csv'), sep=';')

# Nombres equipos
teams_fm = teams_fm_df['team'].unique().tolist()
teams_sw = teams_sw_df['club_name'].unique().tolist()
teams_ss = teams_ss_df['name'].unique().tolist()

# Obtención del dataframe con los equipos unificados por nombre
teams_unified_df = unify_teams(teams_fm, teams_sw, teams_ss, threshold=40)  
fm_team_to_sw = teams_unified_df.set_index('fm')['sw'].to_dict()
ss_team_to_sw = teams_unified_df.set_index('ss')['sw'].to_dict()
sw_team_to_sw = teams_sw_df.set_index('name')['club_name'].to_dict()

In [53]:
# Dataframes de información de los partidos y equipos de ss (necesario para unificar)
ss_matches_df = pd.read_csv(os.path.join(proc_league_path_ss, season, 'match', 'info.csv'), sep=';')
sw_matches_df = pd.read_csv(os.path.join(proc_league_path_sw, season, 'info', 'matches.csv'), sep=';')
teams_ss_df = pd.read_csv(os.path.join(proc_league_path_ss, season, 'info', 'team.csv'), sep=';')

In [81]:
ss = ss_matches_df.copy()
sw = sw_matches_df.copy()
ss_to_sw = ss_team_to_sw
sw_to_sw = sw_team_to_sw

# Nombres de los equipos 
ss['home_team'] = ss['home_team'].map(ss_to_sw)
ss['away_team'] = ss['away_team'].map(ss_to_sw)
sw['home_team'] = sw['home_team'].map(sw_to_sw)
sw['away_team'] = sw['away_team'].map(sw_to_sw)

# Creación de slug del partido
ss_home_slug = ss['home_team'].apply(create_slug)
ss_away_slug = ss['away_team'].apply(create_slug)
sw_home_slug = sw['home_team'].apply(create_slug)
sw_away_slug = sw['away_team'].apply(create_slug)
ss['slug'] = ss_home_slug + '-' + ss_away_slug
sw['slug'] = sw_home_slug + '-' + sw_away_slug

# Concatenamos por slug - usamos left join porque en SS hay los partidos jugados únicamente
matches_df = ss.merge(sw, on='slug', how='left', suffixes=['_ss', '_sw'])

In [82]:
matches_df

,id_ss,slug,round,venue_ss,attendance_ss,referee_ss,home_team_ss,away_team_ss,home_score,away_score,...,home_team_sw,away_team_sw,venue_sw,attendance_sw,match_min,home_score_ht,away_score_ht,home_score_ft,away_score_ft,referee_sw
0,14081792,real_sociedad-real_oviedo,25,Reale Arena,30917.0,José Luis Munuera Montero,Real Sociedad,Real Oviedo,3,3,...,Real Sociedad,Real Oviedo,Estadio Municipal de Anoeta,30917,98,0,0,3,3,José Luis Munuera Montero
1,14081794,osasuna-real_madrid,25,Estadio El Sadar,22480.0,Alejandro Quintero Gonzalez,Osasuna,Real Madrid,2,1,...,Osasuna,Real Madrid,Estadio El Sadar,22480,98,1,0,2,1,Alejandro Quintero González
2,14081796,villarreal-valencia,25,Estadio de la Ceramica,19233.0,Jesus Gil Manzano,Villarreal,Valencia,2,1,...,Villarreal,Valencia,Estadio de la Cerámica,19233,103,2,1,2,1,Jesús Gil Manzano
3,14082849,alaves-levante,1,Estadio de Mendizorroza,12837.0,Miguel Sesma Espinosa,Alavés,Levante,2,1,...,Alavés,Levante,Estadio de Mendizorroza,12837,103,1,0,2,1,Miguel Sesma Espinosa
4,14082850,athletic_club-sevilla,1,San Mamés,49134.0,Francisco Hernandez Maeso,Athletic Club,Sevilla,3,2,...,Athletic Club,Sevilla,Estadio de San Mamés,49134,100,2,0,3,2,Francisco José Hernández Maeso
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
244,14083733,osasuna-celta_de_vigo,10,Estadio El Sadar,20065.0,Alejandro Hernandez,Osasuna,Celta de Vigo,2,3,...,Osasuna,Celta de Vigo,Estadio El Sadar,20065,101,2,1,2,3,Alejandro José Hernández Hernández
245,14083737,espanyol-elche,10,RCDE Stadium,30820.0,Ricardo De Burgos Bengoetxea,Espanyol,Elche,1,0,...,Espanyol,Elche,RCDE Stadium,30820,97,0,0,1,0,Ricardo De Burgos Bengoetxea
246,14083742,mallorca-levante,10,Estadi de Son Moix,12964.0,Miguel Angel Ortiz Arias,Mallorca,Levante,1,1,...,Mallorca,Levante,Estadi Mallorca Son Moix,12964,99,0,1,1,1,Miguel Ángel Ortiz Arias
247,14652471,osasuna-getafe,8,Estadio El Sadar,19895.0,Javier Alberola Rojas,Osasuna,Getafe,2,1,...,Osasuna,Getafe,Estadio El Sadar,19895,100,1,1,2,1,Javier Alberola Rojas


In [49]:
sw_to_sw

{'Athletic Club': 'Athletic Club',
 'CA Osasuna': 'Osasuna',
 'Club Atlético de Madrid': 'Atlético de Madrid',
 'Deportivo Alavés': 'Alavés',
 'Elche CF': 'Elche',
 'FC Barcelona': 'Barcelona',
 'Getafe CF': 'Getafe',
 'Girona FC': 'Girona',
 'Levante UD': 'Levante',
 'Rayo Vallecano de Madrid': 'Rayo Vallecano',
 'Real Betis Balompié': 'Real Betis',
 'Real Club Celta de Vigo': 'Celta de Vigo',
 'Real Club Deportivo Mallorca': 'Mallorca',
 'Real Madrid CF': 'Real Madrid',
 'Real Oviedo': 'Real Oviedo',
 'Real Sociedad de Fútbol': 'Real Sociedad',
 'Reial Club Deportiu Espanyol de Barcelona': 'Espanyol',
 'Sevilla FC': 'Sevilla',
 'Valencia CF': 'Valencia',
 'Villarreal CF': 'Villarreal'}